In [1]:
import yaml
import os
import requests
import pandas as pd
import numpy as np
import logging
from datetime import datetime, timedelta
import alpaca_trade_api as tradeapi

def load_config():
    print("🚀 Attempting to load config.yaml...")  # Debug print
    
    try:
        with open("config.yaml", "r", encoding="utf-8") as file:
            config = yaml.safe_load(file)

        if not config:
            raise ValueError("⚠️ config.yaml is empty or invalid!")

        print("✅ Configuration loaded successfully!")
        return config

    except Exception as e:
        print("❌ Error loading config.yaml:", str(e))
        raise

# Run function and force print
config = load_config()


# Verify installations
print("✅ All necessary packages are imported!")

🚀 Attempting to load config.yaml...
✅ Configuration loaded successfully!
✅ All necessary packages are imported!


In [5]:
# API Configurations
TRADIER_API_KEY = config["tradier"]["api_key"]
ALPACA_API_KEY = config["alpaca"]["api_key"]
ALPACA_API_SECRET = config["alpaca"]["api_secret"]
ALPACA_BASE_URL = config["alpaca"]["base_url"]

# Initialize Alpaca API
alpaca_api = tradeapi.REST(ALPACA_API_KEY, ALPACA_API_SECRET, ALPACA_BASE_URL)

print("✅ API Keys Loaded & Alpaca API Initialized!")

# Define the parent directory (go one level up from "scripts/")
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), os.pardir))

# Define folders outside "scripts/"
DATA_FOLDER = os.path.join(BASE_DIR, "data")
TRADIER_FOLDER = os.path.join(DATA_FOLDER, "tradier")
ALPACA_FOLDER = os.path.join(DATA_FOLDER, "alpaca")
MERGED_FOLDER = os.path.join(DATA_FOLDER, "merged")

# Create folders if they don't exist
for folder in [DATA_FOLDER, TRADIER_FOLDER, ALPACA_FOLDER, MERGED_FOLDER]:
    os.makedirs(folder, exist_ok=True)

print("✅ Folders created successfully!")
print(f"📂 Data Folder: {DATA_FOLDER}")
print(f"📂 Tradier Data Folder: {TRADIER_FOLDER}")
print(f"📂 Alpaca Data Folder: {ALPACA_FOLDER}")
print(f"📂 Merged Data Folder: {MERGED_FOLDER}")

✅ API Keys Loaded & Alpaca API Initialized!
✅ Folders created successfully!
📂 Data Folder: /Users/anbzhnjr/learning/capstone/data
📂 Tradier Data Folder: /Users/anbzhnjr/learning/capstone/data/tradier
📂 Alpaca Data Folder: /Users/anbzhnjr/learning/capstone/data/alpaca
📂 Merged Data Folder: /Users/anbzhnjr/learning/capstone/data/merged


In [6]:
# Load configuration
config = load_config()

# API Credentials
TRADIER_API_KEY = config["tradier"]["api_key"]

# Read values from config
symbols = config["stocks"]
num_days = config["data_settings"]["days_back"]
interval = config["data_settings"]["interval"]

# Define folder paths (outside scripts/)
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
TRADIER_FOLDER = os.path.join(BASE_DIR, "data", "tradier")

# Ensure Tradier folder exists
os.makedirs(TRADIER_FOLDER, exist_ok=True)

# Function to fetch Tradier Data
def fetch_tradier_data(symbol, date):
    url = "https://api.tradier.com/v1/markets/timesales"
    params = {
        "symbol": symbol,
        "interval": interval,
        "start": f"{date} 09:30",
        "end": f"{date} 16:00",
        "session_filter": "open"
    }
    headers = {
        "Authorization": f"Bearer {TRADIER_API_KEY}",
        "Accept": "application/json"
    }
    
    response = requests.get(url, params=params, headers=headers)

    if response.status_code == 200:
        try:
            json_data = response.json()
            return pd.DataFrame(json_data.get("series", {}).get("data", []))
        except Exception as e:
            print(f"⚠️ JSON Parsing Error for {symbol} on {date}: {e}")
            return pd.DataFrame()
    else:
        print(f"⚠️ Tradier API Error ({symbol} - {date}): {response.status_code} - {response.text}")
        return pd.DataFrame()

# Loop through last 60 days and fetch data
for symbol in symbols:
    print(f"📡 Fetching last {num_days} days of data for {symbol}...")

    for i in range(num_days):
        date = (datetime.today() - timedelta(days=i)).strftime("%Y-%m-%d")
        tradier_df = fetch_tradier_data(symbol, date)

        # Define file path
        tradier_file_path = os.path.join(TRADIER_FOLDER, f"{symbol}_{date}_tradier.csv")

        if not tradier_df.empty:
            tradier_df.to_csv(tradier_file_path, index=False)
            print(f"✅ Saved: {tradier_file_path}")
        else:
            print(f"⚠️ No data for {symbol} on {date}")

print("🚀 Data fetching complete!")


🚀 Attempting to load config.yaml...
✅ Configuration loaded successfully!
📡 Fetching last 50 days of data for AAPL...
✅ Saved: /Users/anbzhnjr/learning/capstone/data/tradier/AAPL_2025-03-07_tradier.csv
✅ Saved: /Users/anbzhnjr/learning/capstone/data/tradier/AAPL_2025-03-06_tradier.csv
✅ Saved: /Users/anbzhnjr/learning/capstone/data/tradier/AAPL_2025-03-05_tradier.csv
✅ Saved: /Users/anbzhnjr/learning/capstone/data/tradier/AAPL_2025-03-04_tradier.csv
✅ Saved: /Users/anbzhnjr/learning/capstone/data/tradier/AAPL_2025-03-03_tradier.csv
⚠️ JSON Parsing Error for AAPL on 2025-03-02: 'NoneType' object has no attribute 'get'
⚠️ No data for AAPL on 2025-03-02
⚠️ JSON Parsing Error for AAPL on 2025-03-01: 'NoneType' object has no attribute 'get'
⚠️ No data for AAPL on 2025-03-01
✅ Saved: /Users/anbzhnjr/learning/capstone/data/tradier/AAPL_2025-02-28_tradier.csv
✅ Saved: /Users/anbzhnjr/learning/capstone/data/tradier/AAPL_2025-02-27_tradier.csv
✅ Saved: /Users/anbzhnjr/learning/capstone/data/tradi

In [7]:
# Define paths
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
TRADIER_FOLDER = os.path.join(BASE_DIR, "data", "tradier")
MERGED_FOLDER = os.path.join(BASE_DIR, "data", "merged")
MERGED_FILE_PATH = os.path.join(MERGED_FOLDER, "tradier_merged.csv")

# Ensure merged folder exists
os.makedirs(MERGED_FOLDER, exist_ok=True)

# List all CSV files in Tradier folder
csv_files = [f for f in os.listdir(TRADIER_FOLDER) if f.endswith(".csv")]

# Initialize an empty list to store DataFrames
dataframes = []

# Loop through each CSV file and read into a DataFrame
for file in csv_files:
    file_path = os.path.join(TRADIER_FOLDER, file)
    df = pd.read_csv(file_path)

    # Extract symbol and date from filename (e.g., AAPL_2024-03-06_tradier.csv)
    parts = file.split("_")
    symbol = parts[0]
    date = parts[1]

    # Add symbol & date as new columns in the DataFrame
    df["symbol"] = symbol
    df["date"] = date

    # Append to list
    dataframes.append(df)

# Merge all DataFrames into one
if dataframes:
    merged_df = pd.concat(dataframes, ignore_index=True)
    
    # Save merged data to CSV
    merged_df.to_csv(MERGED_FILE_PATH, index=False, mode="w")
    print(f"✅ Merged CSV saved: {MERGED_FILE_PATH}")
else:
    print("⚠️ No CSV files found to merge!")

# Display first few rows of the merged DataFrame
merged_df.head()


✅ Merged CSV saved: /Users/anbzhnjr/learning/capstone/data/merged/tradier_merged.csv


,time,timestamp,price,open,high,low,close,volume,vwap,symbol,date
0,2025-03-06T09:30:00,1741271400,234.1700,234.435,234.8000,233.54,234.310,1658376,234.36002,AAPL,2025-03-06
1,2025-03-06T09:35:00,1741271700,235.1150,234.240,236.4600,233.77,235.990,1463319,235.47529,AAPL,2025-03-06
2,2025-03-06T09:40:00,1741272000,235.5700,236.005,236.1200,235.02,235.250,959052,235.53971,AAPL,2025-03-06
3,2025-03-06T09:45:00,1741272300,235.2267,235.250,235.6234,234.83,235.145,571212,235.21977,AAPL,2025-03-06
4,2025-03-06T09:50:00,1741272600,235.0075,235.095,235.4450,234.57,234.690,414389,235.01245,AAPL,2025-03-06


In [8]:
import re  # Regex for extracting date

# Define paths
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
TRADIER_FOLDER = os.path.join(BASE_DIR, "data", "tradier")
MERGED_FOLDER = os.path.join(BASE_DIR, "data", "merged")
MERGED_FILE_PATH = os.path.join(MERGED_FOLDER, "tradier_cleaned.csv")
EXCEL_FILE_PATH = os.path.join(BASE_DIR, "data", "merged", "tradier_cleaned.xlsx")

# Ensure merged folder exists
os.makedirs(MERGED_FOLDER, exist_ok=True)

# List all CSV files in Tradier folder
csv_files = [f for f in os.listdir(TRADIER_FOLDER) if f.endswith(".csv")]

# Initialize an empty list to store DataFrames
dataframes = []

# Regex pattern to extract date from filename (e.g., AAPL_2025-01-16_tradier.csv)
date_pattern = re.compile(r"_(\d{4}-\d{2}-\d{2})_tradier\.csv")

# Loop through each CSV file and read into a DataFrame
for file in csv_files:
    file_path = os.path.join(TRADIER_FOLDER, file)
    df = pd.read_csv(file_path)

    match = date_pattern.search(file)
    if match:
        extracted_date = match.group(1)
    else:
        print(f"⚠️ WARNING: Unable to extract date from {file} - Check filename format!")
        extracted_date = "MISSING"

    symbol = file.split("_")[0]  # Extract symbol from filename

    # Add symbol & date as new columns in the DataFrame
    df["symbol"] = symbol
    df["date"] = extracted_date  # Correct date extraction

    # Append to list
    dataframes.append(df)

# Merge all DataFrames into one
if dataframes:
    merged_df = pd.concat(dataframes, ignore_index=True)
    
    # Save merged data to CSV
    merged_df.to_csv(MERGED_FILE_PATH, index=False)
    print(f"✅ Merged CSV saved: {MERGED_FILE_PATH}")
else:
    print("⚠️ No CSV files found to merge!")

# Display first few rows of the merged DataFrame
merged_df.head()

# Confirm unique values in 'date' column
print("\n🔍 Unique values in 'date' column after merging:\n", merged_df["date"].unique())

# Ensure all rows have valid dates
if "UNKNOWN" in merged_df["date"].unique():
    print("\n⚠️ WARNING: Some rows have 'UNKNOWN' as date! Check the filename format.")
else:
    print("\n✅ All rows have correct date values.")

# Save merged data to CSV
merged_df.to_csv(MERGED_FILE_PATH, index=False)
print(f"✅ Merged CSV saved: {MERGED_FILE_PATH}")

# Save to Excel
df.to_excel(EXCEL_FILE_PATH, index=False)

print(f"\n✅ Excel file saved: {EXCEL_FILE_PATH}")

# Display first few rows of the merged DataFrame
from IPython.display import display
display(merged_df.head())



✅ Merged CSV saved: /Users/anbzhnjr/learning/capstone/data/merged/tradier_cleaned.csv

🔍 Unique values in 'date' column after merging:
 ['2025-03-06' '2025-01-17' '2025-01-29' '2025-02-11' '2025-03-04'
 '2025-02-03' '2025-01-30' '2025-02-26' '2025-02-18' '2025-02-04'
 '2025-02-24' '2025-02-14' '2025-01-22' '2025-02-13' '2025-01-27'
 '2025-02-21' '2025-03-03' '2025-02-06' '2025-02-28' '2025-01-23'
 '2025-02-25' '2025-02-05' '2025-02-19' '2025-02-07' '2025-01-31'
 '2025-03-07' '2025-02-12' '2025-01-24' '2025-02-10' '2025-02-20'
 '2025-01-21' '2025-02-27' '2025-01-28' '2025-03-05']

✅ All rows have correct date values.
✅ Merged CSV saved: /Users/anbzhnjr/learning/capstone/data/merged/tradier_cleaned.csv

✅ Excel file saved: /Users/anbzhnjr/learning/capstone/data/merged/tradier_cleaned.xlsx


,time,timestamp,price,open,high,low,close,volume,vwap,symbol,date
0,2025-03-06T09:30:00,1741271400,234.1700,234.435,234.8000,233.54,234.310,1658376,234.36002,AAPL,2025-03-06
1,2025-03-06T09:35:00,1741271700,235.1150,234.240,236.4600,233.77,235.990,1463319,235.47529,AAPL,2025-03-06
2,2025-03-06T09:40:00,1741272000,235.5700,236.005,236.1200,235.02,235.250,959052,235.53971,AAPL,2025-03-06
3,2025-03-06T09:45:00,1741272300,235.2267,235.250,235.6234,234.83,235.145,571212,235.21977,AAPL,2025-03-06
4,2025-03-06T09:50:00,1741272600,235.0075,235.095,235.4450,234.57,234.690,414389,235.01245,AAPL,2025-03-06


In [9]:
# Define paths
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
CLEANED_FILE_PATH = os.path.join(BASE_DIR, "data", "merged", "tradier_cleaned.csv")
EXCEL_SUMMARY_PATH = os.path.join(BASE_DIR, "data", "merged", "tradier_summary.xlsx")

# Load cleaned dataset
df = pd.read_csv(CLEANED_FILE_PATH)

# Select relevant numerical columns for summary statistics
summary_cols = ["open", "high", "low", "close", "volume", "vwap"]

# Compute summary statistics
summary_df = df.groupby("symbol")[summary_cols].agg(["mean", "median", "std"])
summary_df.columns = ["_".join(col) for col in summary_df.columns]  # Flatten MultiIndex

# Compute skewness separately (since `agg` doesn't support it directly)
skewness_df = df.groupby("symbol")[summary_cols].skew()
skewness_df.columns = [f"{col}_skewness" for col in skewness_df.columns]

# Merge summary statistics with skewness
final_summary_df = pd.concat([summary_df, skewness_df], axis=1)

# Save to Excel
final_summary_df.to_excel(EXCEL_SUMMARY_PATH)

print(f"\n✅ Summary statistics saved: {EXCEL_SUMMARY_PATH}")

# Display summary in Jupyter Notebook
from IPython.display import display
display(final_summary_df)




✅ Summary statistics saved: /Users/anbzhnjr/learning/capstone/data/merged/tradier_summary.xlsx


,open_mean,open_median,open_std,high_mean,high_median,high_std,low_mean,low_median,low_std,close_mean,...,volume_std,vwap_mean,vwap_median,vwap_std,open_skewness,high_skewness,low_skewness,close_skewness,volume_skewness,vwap_skewness
symbol,,,,,,,,,,,,,,,,,,,,,
AAPL,235.821508,236.701,7.493266,236.091735,237.02025,7.498023,235.546343,236.405,7.480249,235.830350,...,514231.087669,235.820389,236.688035,7.487858,-0.242027,-0.254363,-0.232152,-0.245164,7.055423,-0.244438
AMZN,224.833625,229.030,12.407503,225.109066,229.25000,12.368395,224.544104,228.850,12.453945,224.833455,...,406304.082567,224.826834,229.021550,12.412525,-0.675335,-0.666954,-0.682017,-0.673040,5.616045,-0.675123
MSFT,413.458121,411.455,16.303376,413.852541,411.79995,16.277267,413.049677,411.110,16.342279,413.463051,...,255342.645147,413.452983,411.435425,16.311134,0.530236,0.543469,0.520400,0.533457,7.360522,0.532068
TSLA,352.946278,356.385,49.892061,353.824106,357.11250,49.821728,351.979858,355.475,49.949409,352.900721,...,680963.024557,352.914387,356.281595,49.890853,-0.356773,-0.351557,-0.361162,-0.355686,2.242218,-0.356273


In [10]:
import pandas as pd
import os

# Define paths
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
CLEANED_FILE_PATH = os.path.join(BASE_DIR, "data", "merged", "tradier_cleaned.csv")

# Load cleaned dataset
df = pd.read_csv(CLEANED_FILE_PATH)

# Check first few rows
print("📊 Initial Data Overview:")
print(df.head())

# Check unique values in the "date" column
print("\n🔍 Unique values in 'date' column:")
print(df["date"].unique())

# Check if all values are "tradier.csv"
if all(df["date"] == "tradier.csv"):
    print("\n🚨 Issue Found: 'date' column contains incorrect values across all rows.")
else:
    print("\n✅ 'date' column has correct values.")


📊 Initial Data Overview:
                  time   timestamp     price     open      high     low  \
0  2025-03-06T09:30:00  1741271400  234.1700  234.435  234.8000  233.54   
1  2025-03-06T09:35:00  1741271700  235.1150  234.240  236.4600  233.77   
2  2025-03-06T09:40:00  1741272000  235.5700  236.005  236.1200  235.02   
3  2025-03-06T09:45:00  1741272300  235.2267  235.250  235.6234  234.83   
4  2025-03-06T09:50:00  1741272600  235.0075  235.095  235.4450  234.57   

     close   volume       vwap symbol        date  
0  234.310  1658376  234.36002   AAPL  2025-03-06  
1  235.990  1463319  235.47529   AAPL  2025-03-06  
2  235.250   959052  235.53971   AAPL  2025-03-06  
3  235.145   571212  235.21977   AAPL  2025-03-06  
4  234.690   414389  235.01245   AAPL  2025-03-06  

🔍 Unique values in 'date' column:
['2025-03-06' '2025-01-29' '2025-02-11' '2025-03-04' '2025-02-03'
 '2025-01-17' '2025-02-26' '2025-02-04' '2025-02-18' '2025-01-30'
 '2025-02-24' '2025-02-14' '2025-01-22' '2025

In [12]:
import numpy as np
import pandas as pd
import os

# Define paths
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
CLEANED_FILE_PATH = os.path.join(BASE_DIR, "data", "merged", "tradier_cleaned.csv")
MATH_FILE_PATH = os.path.join(BASE_DIR, "data", "merged", "tradier_math.csv")

# Load cleaned dataset
df = pd.read_csv(CLEANED_FILE_PATH)

# Ensure 'timestamp' is in datetime format
# Ensure 'timestamp' is correctly parsed
if df["timestamp"].dtype == "int64":  # If it's stored as Unix time (integer)
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s")  # Convert from seconds
else:
    df["timestamp"] = pd.to_datetime(df["timestamp"])  # Convert if it's already a string


# 🔍 Step 1: Check the date column BEFORE any processing
print("\n🔍 Checking `date` column BEFORE transformations:\n", df["date"].unique())

# **Backup Original Date Column Before Any Processing**
df["date"] = df["date"].astype(str)  # Ensure it's a string
date_backup = df[["symbol", "timestamp", "date"]].copy()  # Store date values separately

# Sort data properly
df = df.sort_values(by=["symbol", "timestamp"])

# Compute Daily Returns
df["daily_return"] = df.groupby("symbol")["close"].pct_change()

# Compute Log Returns
df["log_return"] = df.groupby("symbol")["close"].transform(lambda x: np.log(x / x.shift(1)))

# Compute Rolling Volatility (5-day)
df["rolling_volatility"] = df.groupby("symbol")["daily_return"].transform(lambda x: x.rolling(window=5).std())

# Compute Cumulative Returns
df["cumulative_return"] = df.groupby("symbol")["daily_return"].cumsum()

# 🔍 Step 2: Check the date column AFTER calculations
print("\n🔍 Checking `date` column AFTER calculations:\n", df["date"].unique())

# **Step 3: Restore Original Date Column**
df.drop(columns=["date"], inplace=True, errors="ignore")  # Ensure no incorrect date column
df = df.merge(date_backup, on=["symbol", "timestamp"], how="left")  # Restore correct date values

# 🔍 Step 4: Check the date column AFTER restoring
print("\n✅ Final `date` column values after restoration:\n", df["date"].unique())

# **Ensure Overwriting of the File**
if os.path.exists(MATH_FILE_PATH):
    os.remove(MATH_FILE_PATH)  # Delete existing file before writing

# Save to new CSV (overwrite)
df.to_csv(MATH_FILE_PATH, index=False, mode="w")

print(f"\n✅ Math calculations saved (overwritten): {MATH_FILE_PATH}")

# Display sample calculations
from IPython.display import display
display(df.head(10))




🔍 Checking `date` column BEFORE transformations:
 ['2025-03-06' '2025-01-29' '2025-02-11' '2025-03-04' '2025-02-03'
 '2025-01-17' '2025-02-26' '2025-02-04' '2025-02-18' '2025-01-30'
 '2025-02-24' '2025-02-14' '2025-01-22' '2025-02-13' '2025-01-27'
 '2025-02-21' '2025-03-03' '2025-02-06' '2025-02-28' '2025-02-05'
 '2025-02-19' '2025-02-07' '2025-02-25' '2025-01-31' '2025-01-23'
 '2025-03-07' '2025-02-12' '2025-01-24' '2025-02-10' '2025-02-20'
 '2025-01-21' '2025-02-27' '2025-01-28' '2025-03-05']

🔍 Checking `date` column AFTER calculations:
 ['2025-01-17' '2025-01-21' '2025-01-22' '2025-01-23' '2025-01-24'
 '2025-01-27' '2025-01-28' '2025-01-29' '2025-01-30' '2025-01-31'
 '2025-02-03' '2025-02-04' '2025-02-05' '2025-02-06' '2025-02-07'
 '2025-02-10' '2025-02-11' '2025-02-12' '2025-02-13' '2025-02-14'
 '2025-02-18' '2025-02-19' '2025-02-20' '2025-02-21' '2025-02-24'
 '2025-02-25' '2025-02-26' '2025-02-27' '2025-02-28' '2025-03-03'
 '2025-03-04' '2025-03-05' '2025-03-06' '2025-03-07']

✅

,time,timestamp,price,open,high,low,close,volume,vwap,symbol,daily_return,log_return,rolling_volatility,cumulative_return,date
0,2025-01-17T09:30:00,2025-01-17 14:30:00,231.55500,232.115,232.29,230.8200,231.120,11690090,232.12161,AAPL,NaN,NaN,NaN,NaN,2025-01-17
1,2025-01-17T09:35:00,2025-01-17 14:35:00,230.57000,231.190,231.28,229.8600,230.130,1721650,230.54230,AAPL,-0.004283,-0.004293,NaN,-0.004283,2025-01-17
2,2025-01-17T09:40:00,2025-01-17 14:40:00,230.03500,230.120,230.35,229.7200,229.889,1918821,230.08873,AAPL,-0.001047,-0.001048,NaN,-0.005331,2025-01-17
3,2025-01-17T09:45:00,2025-01-17 14:45:00,230.36000,229.870,230.85,229.8700,230.445,1571698,230.53757,AAPL,0.002419,0.002416,NaN,-0.002912,2025-01-17
4,2025-01-17T09:50:00,2025-01-17 14:50:00,230.22000,230.430,230.48,229.9600,230.150,952262,230.20022,AAPL,-0.001280,-0.001281,NaN,-0.004192,2025-01-17
5,2025-01-17T09:55:00,2025-01-17 14:55:00,230.61000,230.150,231.08,230.1400,230.790,1036329,230.60143,AAPL,0.002781,0.002777,0.002927,-0.001412,2025-01-17
6,2025-01-17T10:00:00,2025-01-17 15:00:00,230.92365,230.770,231.24,230.6073,230.920,889866,230.98424,AAPL,0.000563,0.000563,0.001889,-0.000848,2025-01-17
7,2025-01-17T10:05:00,2025-01-17 15:05:00,231.00000,230.915,231.28,230.7200,230.740,646384,231.04231,AAPL,-0.000779,-0.000780,0.001831,-0.001628,2025-01-17
8,2025-01-17T10:10:00,2025-01-17 15:10:00,230.72000,230.750,230.90,230.5400,230.635,777712,230.65126,AAPL,-0.000455,-0.000455,0.001610,-0.002083,2025-01-17
9,2025-01-17T10:15:00,2025-01-17 15:15:00,230.51500,230.620,230.69,230.3400,230.490,505461,230.51469,AAPL,-0.000629,-0.000629,0.001485,-0.002711,2025-01-17


In [10]:
print("\n🔍 Unique symbols in dataset before summary statistics:\n", df["symbol"].unique())
print("Dataset shape:", df.shape)



🔍 Unique symbols in dataset before summary statistics:
 ['AAPL' 'TSLA' 'AMZN' 'MSFT']
Dataset shape: (10608, 11)


In [19]:
import os
import numpy as np
import pandas as pd
from scipy.stats import kurtosis

# Define paths
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
MERGED_FILE_PATH = os.path.join(BASE_DIR, "data", "merged", "tradier_cleaned.csv")
MATH_FILE_PATH = os.path.join(BASE_DIR, "data", "merged", "tradier_math.csv")
SUMMARY_FILE_PATH = os.path.join(BASE_DIR, "data", "merged", "tradier_summary.xlsx")

# 🔹 Load cleaned dataset
df = pd.read_csv(MERGED_FILE_PATH)

# 🔹 Ensure 'timestamp' is in datetime format
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["date"] = df["date"].astype(str)  # Preserve date as string

# 🔹 Backup original date column
original_dates = df[["symbol", "timestamp", "date"]].copy()

# 🔹 Sort data for calculations
df = df.sort_values(by=["symbol", "timestamp"])

# 📌 **Mathematical Calculations**
# Compute Daily Returns
df["daily_return"] = df.groupby("symbol")["close"].pct_change()

# Compute Log Returns (Fix: Reset Index)
df["log_return"] = df.groupby("symbol")["close"].apply(lambda x: np.log(x / x.shift(1))).reset_index(level=0, drop=True)

# Compute Rolling Volatility (Fix: Reset Index)
df["rolling_volatility"] = (
    df.groupby("symbol")["daily_return"]
    .rolling(window=5, min_periods=1)
    .std()
    .reset_index(level=0, drop=True)  # Fix incompatible index issue
)


df["cumulative_return"] = df.groupby("symbol")["daily_return"].cumsum()

# 🔹 Restore date column
df.drop(columns=["date"], inplace=True)
df = df.merge(original_dates, on=["symbol", "timestamp"], how="left")

# 🔹 Save math calculations to CSV (overwrite)
df.to_csv(MATH_FILE_PATH, index=False)
print(f"\n✅ Math calculations saved: {MATH_FILE_PATH}")

# 📌 **Summary Statistics Calculation**
def compute_summary(df):
    summary_stats = df.groupby("symbol").agg({
        "open": ["mean", "median", "std", "skew", "min", "max"],
        "high": ["mean", "median", "std", "skew", "min", "max"],
        "low": ["mean", "median", "std", "skew", "min", "max"],
        "close": ["mean", "median", "std", "skew", "min", "max"],
        "volume": ["mean", "median", "std", "skew", "min", "max"],
        "vwap": ["mean", "median", "std", "skew", "min", "max"]
    })

    # Compute kurtosis separately to avoid lambda issue
    kurtosis_values = df.groupby("symbol")[["open", "high", "low", "close", "volume", "vwap"]].apply(lambda x: x.apply(kurtosis, fisher=True))

    # Flatten column names
    summary_stats.columns = ['_'.join(col).strip() for col in summary_stats.columns]

    # Merge kurtosis values into the summary statistics DataFrame
    for col in kurtosis_values.columns:
        summary_stats[f"{col}_kurtosis"] = kurtosis_values[col]

    return summary_stats

# 🔹 Compute and save summary statistics
summary_stats = compute_summary(df)
summary_stats.to_excel(SUMMARY_FILE_PATH)
print(f"\n✅ Summary statistics saved: {SUMMARY_FILE_PATH}")

# 🔹 Display summary statistics
from IPython.display import display
display(summary_stats)



✅ Math calculations saved: /Users/anbzhnjr/learning/capstone/data/merged/tradier_math.csv

✅ Summary statistics saved: /Users/anbzhnjr/learning/capstone/data/merged/tradier_summary.xlsx


,open_mean,open_median,open_std,open_skew,open_min,open_max,high_mean,high_median,high_std,high_skew,...,vwap_std,vwap_skew,vwap_min,vwap_max,open_kurtosis,high_kurtosis,low_kurtosis,close_kurtosis,volume_kurtosis,vwap_kurtosis
symbol,,,,,,,,,,,,,,,,,,,,,
AAPL,235.821508,236.701,7.493266,-0.242027,219.585,249.5700,236.091735,237.02025,7.498023,-0.254363,...,7.487858,-0.244438,219.66352,249.61355,-0.851602,-0.841960,-0.854499,-0.848361,101.918197,-0.848076
AMZN,224.833625,229.030,12.407503,-0.675335,192.975,242.4800,225.109066,229.25000,12.368395,-0.666954,...,12.412525,-0.675123,192.88185,242.41264,-0.750822,-0.764488,-0.743349,-0.756768,60.716368,-0.752874
MSFT,413.458121,411.455,16.303376,0.530236,382.130,448.3300,413.852541,411.79995,16.277267,0.543469,...,16.311134,0.532068,382.24033,448.20288,-0.465676,-0.471168,-0.460767,-0.465168,102.383475,-0.468084
TSLA,352.946278,356.385,49.892061,-0.356773,251.880,439.4799,353.824106,357.11250,49.821728,-0.351557,...,49.890853,-0.356273,251.52757,439.44448,-0.989316,-0.991150,-0.990153,-0.991814,6.933865,-0.991176
